In [2]:
import cvxpy as cp
import numpy as np

# --- Dados do Sistema ---
n_usinas = 3
n_linhas = 3
n_cargas = 3

# Custo incremental das usinas ($/MVAh)
# Usina 1 é a mais barata, Usina 3 a mais cara
c = np.array([10, 20, 30])

# Capacidade máxima das usinas (MVA)
P_max = np.array([100, 150, 200])

# Capacidade das linhas de transmissão (MVA)
F_max = np.array([50, 70, 80])

# Demanda das cargas (MVA)
# Alterado para concentrar carga na Barra 3, forçando o fluxo vindo da Barra 1 e 2
D = np.array([20, 30, 170])

# Matriz de incidência das linhas (Topologia da rede)
# L1: 1->2, L2: 1->3, L3: 2->3
A = np.array([
    [1, -1, 0],  # Linha 1 conecta Barra 1 e 2
    [1, 0, -1],  # Linha 2 conecta Barra 1 e 3
    [0, 1, -1]   # Linha 3 conecta Barra 2 e 3
])

# --- Variáveis de Otimização ---
P = cp.Variable(n_usinas) # Geração (MVA)
F = cp.Variable(n_linhas) # Fluxo (MVA)
theta = cp.Variable(n_cargas) # Ângulo (rad)

# --- Função Objetivo ---
# Minimizar o custo total de geração
obj = cp.Minimize(c @ P)

# --- Restrições ---
constraints = [
    cp.sum(P) == cp.sum(D),       # Balanço Global de Potência
    P <= P_max,                   # Limite de Geração
    P >= 0,                       # Geração não-negativa
    F == A @ theta,               # Lei de Kirchhoff para Fluxo DC
    F <= F_max,                   # Limite térmico da linha (Direto)
    F >= -F_max,                  # Limite térmico da linha (Reverso)
    theta[0] == 0                 # Barra de Referência (Slack Bus)
]

# Adicionando o Balanço de Carga por Barra (Lei de Kirchhoff das Correntes)
# Geração - Carga = Fluxo que sai da barra
constraints += [P - D == A.T @ F]

# --- Solução do Problema ---
prob = cp.Problem(obj, constraints)
prob.solve()

# --- Exibição dos Resultados ---
if prob.status == 'optimal':
    print(f"Status: {prob.status}")
    print(f"Custo Total: {prob.value:.2f} $/hora\n")

    print("Geração das Usinas (MVA):")
    for i in range(n_usinas):
        print(f"  Usina {i+1}: {P.value[i]:.1f} MVA")

    print("\nFluxo nas Linhas de Transmissão (MVA):")
    print(f"  Linha 1 (Barra 1-2): {F.value[0]:.1f} MVA")
    print(f"  Linha 2 (Barra 1-3): {F.value[1]:.1f} MVA")
    print(f"  Linha 3 (Barra 2-3): {F.value[2]:.1f} MVA")

    print("\nÂngulo das Tensões (rad):")
    for i in range(n_cargas):
        print(f"  Barra {i+1}: {theta.value[i]:.4f} rad")
else:
    print("O problema não encontrou uma solução ótima. Verifique as restrições de F_max.")

Status: optimal
Custo Total: 3800.00 $/hora

Geração das Usinas (MVA):
  Usina 1: 97.9 MVA
  Usina 2: 84.2 MVA
  Usina 3: 37.9 MVA

Fluxo nas Linhas de Transmissão (MVA):
  Linha 1 (Barra 1-2): 7.9 MVA
  Linha 2 (Barra 1-3): 70.0 MVA
  Linha 3 (Barra 2-3): 62.1 MVA

Ângulo das Tensões (rad):
  Barra 1: -0.0000 rad
  Barra 2: -7.9039 rad
  Barra 3: -70.0000 rad
